# Week 06: Stats for Business: Is the Price Right?

You just joined the pricing team at a car-listing marketplace. Your manager has a few questions and one rule: **every claim needs a number, and every number needs a reason to trust it.**

**Data:** `cars.csv`: about 12,000 car trims (1990–2017) with specs and sticker price (MSRP). Source: [Car Features and MSRP (Kaggle)](https://www.kaggle.com/datasets/CooperUnion/cardataset). It's real data, so it's messy.

**What you'll practice** (all from this week's lecture):
- Cleaning before describing
- Mean vs median, skew, log transforms
- z-scores
- Bootstrap confidence intervals
- One- and two-sample tests, and picking the right one
- Confounding, and significant vs important

**How to answer:** Fill in the code cells, then write 1–3 sentences in each **Answer:** cell. Code without an interpretation is only half the job.

Plan for about 2–3 hours. 🚀 questions are optional stretch.

## Setup

In [ ]:
# run this first if you dont have them installed in your env
!pip install pandas numpy matplotlib scipy

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from pathlib import Path

pd.set_option("display.max_columns", None)
rng = np.random.default_rng(42)

data_path = Path("data/cars.csv") if Path("data/cars.csv").exists() else Path("../data/cars.csv")
df = pd.read_csv(data_path)
print(df.shape)
df.head()


---
## Part 1: Can you trust this data?

### Q1. Find the problems before you average anything

This dataset has **at least five** real problems. Find them, fix or flag each one, and build a cleaned DataFrame called `clean`. You'll use `clean` for the rest of the notebook.

Use your Week 02 cleaning checklist. Some nudges, in no particular order:
- Count things twice.
- The most common value in a column can be more suspicious than the largest one.
- Would you print every max value in a sales brochure?
- Missing values usually have a reason. *Who* is missing?
- Is every number in a column measured in the same unit?
- Does every column vary at the level you'd expect?
- Not everything weird is wrong.

For each problem, decide: **drop it, set it to NaN, flag it, or leave it**. Dropping whole rows is the last resort, because a row with a bad price may still have a good MPG.

In [ ]:
# Initial checks used to locate data quality problems.
print("duplicate rows:", df.duplicated().sum())
print("MSRP value counts (smallest prices):", df["MSRP"].value_counts().head())
print("highway MPG extremes:", df["highway MPG"].nlargest(5).tolist())
print("unknown transmissions:", (df["Transmission Type"] == "UNKNOWN").sum())
print("missing values:\n", df.isna().sum()[df.isna().sum() > 0])
print("zero-cylinder rows:", (df["Engine Cylinders"] == 0).sum())


**Cleaning log:**

| # | Problem | How I found it | Rows affected | What I did | Why |
|---|---|---|---|---|---|
| 1 | Exact duplicate rows | `df.duplicated().sum()` | 720 | Dropped duplicate rows | They repeat the same trim and would give it extra weight. |
| 2 | MSRP placeholder of 2000 | `MSRP.value_counts()` showed 1,036 repeats | 1,036 | Set MSRP 2000 to NaN | It is an encoded missing value for older vehicles, not a comparable sticker price. |
| 3 | Impossible highway MPG of 354 | `highway MPG.nlargest()` | 1 | Set highway MPG 354 to NaN | It is a transcription/unit error and would distort MPG summaries. |
| 4 | Unknown transmission category | `Transmission Type.value_counts()` | 19 before deduplication | Set UNKNOWN to NaN | It cannot be assigned to automatic or manual comparisons. |
| 5 | Missing/zero engine fields | `isna().sum()` and `Engine Cylinders.value_counts()` | 69 HP missing; 30 cylinders missing; 56 zero cylinders | Keep missing values as NaN; set 0 cylinders to NaN | Missing specifications should not be invented; zero cylinders is not a valid combustion-engine count. |


In [ ]:
# Build your cleaned DataFrame. Start from a copy so df stays untouched.
clean = df.copy()
clean.columns = clean.columns.str.strip().str.lower().str.replace(" ", "_")
clean = clean.drop_duplicates().reset_index(drop=True)
clean.loc[clean["msrp"] == 2000, "msrp"] = np.nan
clean.loc[clean["highway_mpg"] == 354, "highway_mpg"] = np.nan
clean.loc[clean["transmission_type"] == "UNKNOWN", "transmission_type"] = np.nan
clean.loc[clean["engine_cylinders"] == 0, "engine_cylinders"] = np.nan


In [ ]:
# Checkpoint: run this before moving on.
print(clean.shape)
print(clean.isna().sum()[clean.isna().sum() > 0])
# Fewer than ~10,000 rows? You probably dropped too much. Go back and use NaN or a flag instead.

---
## Part 2: Describe

### Q2. What does a "typical" car cost?

Your manager wants one number for "typical price" on the homepage.

a) Compute the mean and median MSRP. Which is bigger, and why?

b) Plot a histogram of MSRP, then a histogram of `np.log10(msrp)`. Describe each shape in a few words.

c) What percent of cars cost more than the mean?

d) Which number belongs on the homepage? Why?

In [ ]:
# a) mean and median
prices = clean["msrp"].dropna()
mean_msrp, median_msrp = prices.mean(), prices.median()
print(f"Mean MSRP: ${mean_msrp:,.2f}")
print(f"Median MSRP: ${median_msrp:,.2f}")

# b) two histograms side by side
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(prices, bins=50, edgecolor="white")
axes[0].set(title="MSRP", xlabel="Dollars", ylabel="Cars")
axes[1].hist(np.log10(prices), bins=40, edgecolor="white")
axes[1].set(title="log10(MSRP)", xlabel="log10 dollars", ylabel="Cars")
plt.tight_layout()
plt.show()

# c) share of cars above the mean
share_above_mean = (prices > mean_msrp).mean() * 100
print(f"Percent above the mean: {share_above_mean:.1f}%")


**Answer:** The mean is $44,788.67 and the median is $31,870; the mean is higher because a small number of luxury and exotic cars create a long right tail. The raw histogram is strongly right-skewed, while the log10 histogram is much more balanced. Only 24.4% of cars are above the mean, so I would put the median ($31,870) on the homepage because it better represents a typical listing.


### Q3. Fuel-efficient *for its size*?

Two 2017 cars:
- **Honda Civic** (Compact): 42 highway MPG
- **Volvo S90** (Large): 34 highway MPG

The Civic wins on raw MPG, but large cars are heavier. Which one is more impressive *for its class*?

a) For 2017 cars (only the MPG values you trust after Q1), compute the mean and standard deviation of highway MPG for each `vehicle_size`.

b) Compute each car's z-score within its own size class.

c) Which is more fuel-efficient for its class? How unusual is each one?

In [ ]:
# a) mean and std of trusted highway MPG by vehicle_size, 2017 only
cars_2017 = clean[clean["year"] == 2017].dropna(subset=["highway_mpg"])
mpg_by_size = cars_2017.groupby("vehicle_size")["highway_mpg"].agg(["count", "mean", "std"])
print(mpg_by_size)

# b) z = (value - class mean) / class std
z_civic = (42 - mpg_by_size.loc["Compact", "mean"]) / mpg_by_size.loc["Compact", "std"]
z_volvo = (34 - mpg_by_size.loc["Large", "mean"]) / mpg_by_size.loc["Large", "std"]
print(f"Honda Civic z-score: {z_civic:.2f}")
print(f"Volvo S90 z-score: {z_volvo:.2f}")


**Answer:** In 2017, compact cars averaged 31.99 MPG (SD 10.78) and large cars averaged 22.99 MPG (SD 3.49). The Civic is 0.93 standard deviations above its compact-class mean, while the S90 is 3.15 standard deviations above its large-class mean, so the S90 is more unusual and more impressive for its class.


---
## Part 3: Uncertainty

### Q4. How sure are we about the median price?

The 2017 cars are one snapshot of the market. With a slightly different set of cars, the median would move. A bootstrap shows how much.

a) Compute the median MSRP of 2017 cars.

b) Bootstrap it: resample the 2017 prices **with replacement** 5,000 times and take the median each time. Plot the 5,000 medians and report the 95% interval (2.5th and 97.5th percentiles).

c) Repeat for 2017 cars from **one make** with fewer than 50 cars that year (your choice). Is the interval wider or narrower? Why?

d) Explain the interval from (b) in one sentence a manager would understand.

In [ ]:
def bootstrap_median(values, n_boot=5000):
    """Return an array of n_boot bootstrap medians."""
    values = np.asarray(values)
    samples = rng.choice(values, size=(n_boot, len(values)), replace=True)
    return np.median(samples, axis=1)

cars_2017 = clean[clean["year"] == 2017]
prices_2017 = cars_2017["msrp"].dropna().to_numpy()
median_2017 = np.median(prices_2017)
boot_2017 = bootstrap_median(prices_2017)
ci_2017 = np.percentile(boot_2017, [2.5, 97.5])
print(f"2017 median: ${median_2017:,.2f}; 95% bootstrap interval: ${ci_2017[0]:,.2f} to ${ci_2017[1]:,.2f}")
plt.hist(boot_2017, bins=40, edgecolor="white")
plt.title("Bootstrap medians: 2017 MSRP")
plt.xlabel("Median MSRP")
plt.show()

# Subaru has fewer than 50 2017 rows.
make = "Subaru"
make_prices = cars_2017.loc[cars_2017["make"] == make, "msrp"].dropna().to_numpy()
boot_make = bootstrap_median(make_prices)
ci_make = np.percentile(boot_make, [2.5, 97.5])
print(f"{make} n={len(make_prices)}, median=${np.median(make_prices):,.2f}; 95% interval: ${ci_make[0]:,.2f} to ${ci_make[1]:,.2f}")


**Answer:** The 2017 median MSRP is $36,737.50, with a 95% bootstrap interval of about $35,940–$37,683. Subaru has only 46 cars, giving a wider interval ($24,045–$28,840) because a smaller sample gives a less precise median. In manager terms, if we repeatedly sampled comparable 2017 listings, the typical median would usually fall in the reported interval.


---
## Part 4: Testing

### Q5. Can marketing say "our 2017 compacts average over 30 MPG"?

a) Write H₀ and Hₐ. One-sided or two-sided? Why?

b) How many cars are in the sample, and what does the distribution look like? Is a t-test reasonable here?

c) Run a one-sample t-test at α = 0.05 (`stats.ttest_1samp(..., alternative="greater")`). State the conclusion in plain English.

d) **Independence check.** Tests assume each row is an independent observation. Count how many rows each `make` + `model` has in this sample. Then average to one row per model and rerun the test. What changed, and which version do you trust more?

In [ ]:
# b) sample: 2017 compact cars (MPG you trust); n, mean, histogram
compact_2017 = clean[(clean["year"] == 2017) & (clean["vehicle_size"] == "Compact")].dropna(subset=["highway_mpg"])
mpg_sample = compact_2017["highway_mpg"]
print(f"n={len(mpg_sample)}, mean={mpg_sample.mean():.2f}, SD={mpg_sample.std():.2f}")
plt.hist(mpg_sample, bins=20, edgecolor="white")
plt.xlabel("Highway MPG"); plt.ylabel("Cars"); plt.title("2017 compact highway MPG"); plt.show()

# c) one-sample t-test vs 30: H0: mu <= 30; Ha: mu > 30.
test_rows = stats.ttest_1samp(mpg_sample, 30, alternative="greater")
print("row-level test:", test_rows)

# d) rows per make+model, then one row per model and retest
print(compact_2017.groupby(["make", "model"]).size().describe())
model_mpg = compact_2017.groupby(["make", "model"])["highway_mpg"].mean()
model_test = stats.ttest_1samp(model_mpg, 30, alternative="greater")
print(f"one row per model: n={len(model_mpg)}, mean={model_mpg.mean():.2f}, test={model_test}")


**Answer:** H₀ is μ ≤ 30 MPG and Hₐ is μ > 30 MPG, so this is a one-sided test because the claim has a specific direction. The 517-row sample averages 31.99 MPG; the t-test gives p = 0.000015, supporting an average above 30 MPG. However, trims are repeated within make/model, so the model-level test is more defensible: 90 models average 35.62 MPG with p = 0.00131, and it still supports the claim while avoiding pseudo-replication.


### Q6. Do manual cars cost less? (the big one)

A sales rep says: *"Manual cars are way cheaper. We should push manuals to budget shoppers."*

Compare `AUTOMATIC` vs `MANUAL` only.

a) Compare the median MSRP of the two groups.

b) Before believing it, compare the `year` of manual vs automatic cars. What do you notice?

c) Redo (a) using only 2015–2017 cars, then within each `vehicle_size`. For 2015–17 compacts, use the Mann–Whitney U test (`stats.mannwhitneyu(auto, manual)`) to compare prices.

- What do the p-value and medians suggest?

d) Write one sentence back to the sales rep.

In [ ]:
# a) compare median MSRP by transmission type
am = clean[clean["transmission_type"].isin(["AUTOMATIC", "MANUAL"])]
print("overall medians\n", am.groupby("transmission_type")["msrp"].median())

# b) compare year by transmission type
print("year medians\n", am.groupby("transmission_type")["year"].median())

# c) compare 2015-2017 overall and within each vehicle_size
recent = am[am["year"].between(2015, 2017)]
print("2015-17 medians\n", recent.groupby("transmission_type")["msrp"].median())
print("2015-17 by size\n", recent.groupby(["vehicle_size", "transmission_type"])["msrp"].median().unstack())
compact_recent = recent[recent["vehicle_size"] == "Compact"]
auto = compact_recent.loc[compact_recent["transmission_type"] == "AUTOMATIC", "msrp"].dropna()
manual = compact_recent.loc[compact_recent["transmission_type"] == "MANUAL", "msrp"].dropna()
print("compact Mann-Whitney U:", stats.mannwhitneyu(auto, manual, alternative="two-sided"))


**Answer:** Across all years, automatic cars have a median MSRP of $33,620 versus $21,875 for manuals, but manuals are also older (median year 2008 versus 2015), so the raw gap is confounded by age. In 2015–2017 the medians are $37,065 automatic and $26,905 manual; within compacts they are $25,995 and $25,860, and the Mann–Whitney p-value is 0.205. I would tell the sales rep that manuals are not demonstrably cheaper among recent compact cars; the overall difference mostly reflects which years and vehicle classes contain manuals.


### Q7. Significant ≠ important

Among cars whose MPG you trust, 4-door cars get better highway MPG than 2-door cars.

a) Run a Welch t-test. Report the difference in means and the p-value.

b) Translate the difference into dollars. Assume 12,000 miles/year and \$3.50/gallon. How much does the average 4-door driver save per year?

c) Now pretend you only had **30 cars per group.** Draw 30 random 4-door and 30 random 2-door cars, run the same test, and repeat 1,000 times. What fraction of those tests come out significant (p < 0.05)?

d) Would you put "4-door cars are significantly more fuel-efficient" in an ad? What would you say instead?

In [ ]:
# a) Welch t-test, 4-door vs 2-door
four = clean.loc[clean["number_of_doors"] == 4, "highway_mpg"].dropna()
two = clean.loc[clean["number_of_doors"] == 2, "highway_mpg"].dropna()
welch = stats.ttest_ind(four, two, equal_var=False)
diff_mpg = four.mean() - two.mean()
print(f"4-door mean={four.mean():.2f}, 2-door mean={two.mean():.2f}, difference={diff_mpg:.2f}, p={welch.pvalue:.3g}")

# b) yearly gas-cost difference at 12,000 miles and $3.50/gallon
annual_savings = 12000 * 3.50 * (1 / two.mean() - 1 / four.mean())
print(f"estimated annual savings for the average 4-door driver: ${annual_savings:,.2f}")

# c) simulation
n_sims, hits = 1000, 0
for _ in range(n_sims):
    sample_four = rng.choice(four.to_numpy(), 30, replace=False)
    sample_two = rng.choice(two.to_numpy(), 30, replace=False)
    if stats.ttest_ind(sample_four, sample_two, equal_var=False).pvalue < 0.05:
        hits += 1
print(f"significant simulations: {hits}/{n_sims} = {hits/n_sims:.3f}")


**Answer:** Four-door cars average 27.33 MPG versus 25.34 for two-door cars, a 1.99 MPG difference with p < 0.001. At 12,000 miles and $3.50/gallon, that is about $120.78 per year in average fuel savings. With only 30 cars per group, the test was significant in 13.4% of simulations, so I would describe the observed association and the modest dollar impact rather than use “significantly more fuel-efficient” as an unconditional ad claim.


---
## Part 5: Communicate

### Q8. Your memo to the pricing manager

Write 4–6 sentences. Include:
- Your answer to "do manual cars cost less?", with at least one number
- One other finding from this notebook that matters for pricing
- One limitation of this data a manager should know about

No unexplained jargon: if you say "significant," say what it means.

**Memo:** Manual listings have a lower overall median MSRP ($21,875 versus $33,620), but they are much older (median 2008 versus 2015), so age and vehicle mix confound that comparison. For recent 2015–2017 compact cars, the medians are nearly identical ($25,860 manual versus $25,995 automatic) and the Mann–Whitney p-value is 0.205. A useful pricing benchmark is the overall median MSRP of $31,870, because the price distribution is strongly right-skewed by luxury and exotic vehicles. The data contain duplicate trim rows and placeholder prices, and the remaining rows are listings rather than a random sample of all cars, so these results describe this dataset rather than proving causal effects.


---
## 🚀 Stretch (optional)

### 🚀 S1. Horsepower and price

a) Compute the correlation between `engine_hp` and `msrp` three ways: Pearson, Pearson with `np.log10(msrp)`, and Spearman. Why do they differ?

b) Scatter plot HP vs MSRP with a log y-axis (`plt.yscale("log")`).

c) Does adding horsepower *cause* a higher price? Name one lurking variable.

In [ ]:
# S1: horsepower and price correlations
s1 = clean[["engine_hp", "msrp"]].dropna()
print("Pearson HP vs MSRP:", stats.pearsonr(s1["engine_hp"], s1["msrp"]))
print("Pearson HP vs log10(MSRP):", stats.pearsonr(s1["engine_hp"], np.log10(s1["msrp"])))
print("Spearman HP vs MSRP:", stats.spearmanr(s1["engine_hp"], s1["msrp"]))
plt.scatter(s1["engine_hp"], s1["msrp"], s=8, alpha=.25)
plt.yscale("log"); plt.xlabel("Engine horsepower"); plt.ylabel("MSRP (log scale)"); plt.title("Horsepower and MSRP"); plt.show()


### 🚀 S2. Is transmission tied to vehicle size?

For 2015–2017 cars (automatic and manual only), build a crosstab of `vehicle_size` × `transmission_type` and run a chi-square test (`stats.chi2_contingency`). How does this connect to Q6?

In [ ]:
# S2: transmission and vehicle size, 2015-2017
s2 = clean[clean["year"].between(2015, 2017) & clean["transmission_type"].isin(["AUTOMATIC", "MANUAL"])]
table = pd.crosstab(s2["vehicle_size"], s2["transmission_type"])
print(table)
print(stats.chi2_contingency(table))
